<a href="https://colab.research.google.com/github/wandb/examples/blob/master/colabs/automations/automations-tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<!--- @wandbcode{automations-tutorial} -->

# W&B Automations tutorial (API)

This notebook walks through **W&B Automations** with the Python API. Run the notebook to configure your team, list automations, and create a project automation that will send a message to a Slack channel when a run in the project fails. Simulate a failure to test the automation, optionally update the automation for further testing, and delete it when testing is complete.

For full documentation, including instructions for managing automations in the W&B App, see the [project automation tutorial](https://docs.wandb.ai/models/automations/project-automation-tutorial) and [registry automation tutorial](https://docs.wandb.ai/models/automations/registry-automation-tutorial).

## Prerequisites

- Install the W&B Python SDK and log in (next cells).
- A W&B **team** (entity) with a **project**.
- **Team Settings**: a [Slack integration](https://docs.wandb.ai/models/automations/create-automations/slack) and a [webhook](https://docs.wandb.ai/models/automations/create-automations/webhook).
- Edit **Configuration** with your `ENTITY` and `PROJECT` details.

In [ ]:
!pip install wandb -qU

In [ ]:
import wandb

# Log in to your W&B account (same pattern as the intro Colab notebooks)
wandb.login()

## Configuration

Edit these values to match your team, project, and registry collection.

In [ ]:
# Team or user entity (visible in your W&B URL)
ENTITY = "my-team"

# Project used for the run-failure example and for logging a test artifact
PROJECT = "my-project"

api = wandb.Api()

## List automations

Lists automations for your entity. The list may be empty if you have not created any yet.

In [ ]:
for automation in api.automations(entity=ENTITY):
    print(automation.name, "|", getattr(automation, "scope", ""), "|", getattr(automation, "enabled", True))

## Create: Project automation (run failure to Slack)

Aligns with the [project automation tutorial](https://docs.wandb.ai/models/automations/project-automation-tutorial) in the W&B documentation.

In [ ]:
from wandb.automations import OnRunState, RunEvent, SendNotification

project = api.project(PROJECT, entity=ENTITY)
slack_integration = next(api.slack_integrations(entity=ENTITY))

event = OnRunState(
    scope=project,
    filter=RunEvent.state.in_(["failed"]),
)
action = SendNotification.from_integration(slack_integration)

project_automation = api.create_automation(
    event >> action,
    name="run-failure-alert",
    description="Notify the team when a run fails.",
)
print("Created:", project_automation.name)

## Optional: Test the project automation

Run this cell to log a failed run in the same project you used above. Within a short time you should see a Slack message with the run link and status.

In [ ]:
with wandb.init(project=PROJECT, entity=ENTITY) as run:
    run.log({"loss": 1.23})
    run.finish(exit_code=1)

## Get one automation by name

Use this after you have created an automation. If the name does not exist or is ambiguous, the API raises `ValueError`. See also [Manage automations with the API](https://docs.wandb.ai/models/automations/api#get-one-automation).

In [ ]:
fetched = api.automation(name="run-failure-alert", entity=ENTITY)
print(fetched.name, fetched.description, fetched.enabled)

## Update an automation

Run the **optional** test cells above before this section if you want Slack or webhook delivery while the automations are still enabled.

This section fetches the project automation by name, then disables it and updates the description.

In [ ]:
to_update = api.automation(name="run-failure-alert", entity=ENTITY)
updated = api.update_automation(
    to_update,
    enabled=False,
    description="Temporarily disabled.",
)
print("Updated:", updated.name, updated.enabled, updated.description)

## Delete an automation

Optionally, run this cell to delete the automation when you are finished testing.

In [ ]:
api.delete_automation(api.automation(name="run-failure-alert", entity=ENTITY))